# T1 - Transducer (standalone)
T1 keeps a registry of piezoelectric materials and a valid frequency range for each (`process_material`). Below: the team's code as it is, then the probe geometry the integrated v2 pipeline uses (128 elements, 5 MHz, pitch = lambda/2, defined in `T3`'s config).

In [ ]:
# --- Step 0: find the project root (works from any folder: VS Code, Jupyter, Colab) and add every module to sys.path
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_here, *_here.parents] if (p / "integration" / "03_full_pipeline" / "_paths.py").exists()), None)
if ROOT is None:
    raise SystemExit("Project root not found. Open the NITK-UsoundSim-v2 folder in VS Code (File > Open Folder) and run again.")
sys.path.insert(0, str(ROOT / "integration" / "03_full_pipeline"))
import _paths                                   # noqa: F401  (adds modules/01..05 and integration/01..03 to sys.path)
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline") if "ipykernel" in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np
print("project root:", ROOT)

In [ ]:
sys.path.insert(0, str(ROOT / 'teams' / 'T1_Transducer' / 'src'))
from registry import registry
from process import process_material
from exceptions import PiezoValidationError, PiezoNotFoundError

## 1. Register materials and draw a centre frequency from the allowed range

In [ ]:
registry.clear()
for name, rng in [('Quartz', (100.0, 500.0)), ('PZT-5H', (1000.0, 5000.0))]:
    f = process_material(name, rng, create_new=True)
    print(f'{name:8s} range {rng} kHz -> drawn {f:8.2f} kHz')
print('registered:', registry.list_materials())
print('lookup Quartz again:', round(process_material('Quartz', create_new=False), 2), 'kHz')

## 2. Validation errors are raised, not hidden

In [ ]:
for bad in (lambda: process_material('BadRange', (500.0, 1000.0 - 2000.0), create_new=True), lambda: process_material('Unknown', create_new=False)):
    try: bad()
    except (PiezoValidationError, PiezoNotFoundError) as e: print(type(e).__name__, '->', e)

## 3. The probe used by the integrated pipeline
Geometry comes from the updated T3 configuration: 128 elements, 5 MHz, pitch lambda/2.

In [ ]:
from nitk_usoundsim import config
x = (np.arange(config.N_ELEMENTS) - (config.N_ELEMENTS - 1) / 2) * config.PITCH
print(f'{config.N_ELEMENTS} elements | f0 {config.F0/1e6:g} MHz | pitch {config.PITCH*1e3:.3f} mm | aperture {(x[-1]-x[0])*1e3:.1f} mm')
plt.figure(figsize=(9, 1.6)); plt.scatter(x*1e3, np.zeros_like(x), marker='s', s=14); plt.yticks([]); plt.xlabel('element x (mm)'); plt.title('element positions'); plt.show()